In [18]:
import os
import xarray as xr
import pandas as pd
import numpy as np

In [ ]:
nc_dir = "musica_data"
all_files = os.listdir(nc_dir)
nc_files = [os.path.join(nc_dir, f) for f in all_files if f.lower().endswith('.nc')]
nc_files = sorted(nc_files)

print(f"--- [Status] Scan Completed ---")
print(f"Total .nc files detected in cluster storage: {len(nc_files)}")

file_info = [{"Index": i+1, "NC Filename": os.path.basename(f)} for i, f in enumerate(nc_files)]
df_files = pd.DataFrame(file_info)

print("\n>>> Detected MUSICA Dataset Table:")
display(df_files)




--- [Status] Scan Completed ---
Total .nc files detected in cluster storage: 15

>>> Detected MUSICA Dataset Table:


,Index,NC Filename
0,1,MUSICA_production3.2_BB.cam.h2.2018-07-01-0360...
1,2,MUSICA_production3.2_BB.cam.h2.2018-07-02-0360...
2,3,MUSICA_production3.2_BB.cam.h2.2018-07-03-0360...
3,4,MUSICA_production3.2_BB.cam.h2.2018-07-04-0360...
4,5,MUSICA_production3.2_BB.cam.h2.2018-07-05-0360...
5,6,MUSICA_production3.2_BB.cam.h2.2018-07-06-0360...
6,7,MUSICA_production3.2_BB.cam.h2.2018-07-07-0360...
7,8,MUSICA_production3.2_BB.cam.h2.2018-07-08-0360...
8,9,MUSICA_production3.2_BB.cam.h2.2018-07-09-0360...
9,10,MUSICA_production3.2_BB.cam.h2.2018-07-10-0360...


In [19]:
print("\n--- FIRST MUSICA DATASET INFO (METADATA) ---")
ds_preview = xr.open_dataset(nc_files[0])
print(ds_preview)
ds_preview.close()


--- FIRST MUSICA DATASET INFO (METADATA) ---
<xarray.Dataset>
Dimensions:       (ncol: 92018, lev: 32, ilev: 33, time: 24, nbnd: 2)
Coordinates:
  * lev           (lev) float64 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * ilev          (ilev) float64 2.255 5.032 10.16 18.56 ... 967.5 985.1 1e+03
  * time          (time) datetime64[ns] 2018-07-01T01:00:00 ... 2018-07-02
Dimensions without coordinates: ncol, nbnd
Data variables: (12/34)
    lat           (ncol) float64 ...
    lon           (ncol) float64 ...
    area          (ncol) float64 ...
    hyam          (lev) float64 ...
    hybm          (lev) float64 ...
    hyai          (ilev) float64 ...
    ...            ...
    PM25          (time, lev, ncol) float32 ...
    PRECT         (time, ncol) float32 ...
    RELHUM        (time, lev, ncol) float32 ...
    T             (time, lev, ncol) float32 ...
    U             (time, lev, ncol) float32 ...
    V             (time, lev, ncol) float32 ...
Attributes:
    ne:          

In [ ]:
# Extract UK-domain fields from MUSICA NetCDF files
from xarray.backends import NetCDF4BackendEntrypoint

# List NetCDF files in the MUSICA archive
data_dir = "/mnt/iusers01/msc-stu/hum-msc-data-sci-2025-2026/g05844jc/scratch/musica_data/MUSICA"
all_files = os.listdir(data_dir)
nc_files = [os.path.join(data_dir, f) for f in all_files if f.lower().endswith('.nc') and "slurm" not in f.lower()]
nc_files = sorted(nc_files)

# UK geographic bounding box
lon_min, lon_max = -9.0, 1.8
lat_min, lat_max = 49.0, 61.0

var_list = ['lon', 'lat', 'area', 'PM25', 'PBLH', 'T', 'U', 'V']

processed_dataframes = []
print("\n--- [Step 2] Commencing Spatial Extraction for UK (Comprehensive Variables) ---")

backend = NetCDF4BackendEntrypoint()

for idx, file_path in enumerate(nc_files):
    filename = os.path.basename(file_path)
    
    # Open with the NetCDF4 backend to avoid engine conflicts and thread locks
    ds = backend.open_dataset(file_path)
    
    # Resolve the longitude and latitude variable names
    lat_name = [v for v in ds.variables if 'lat' in v.lower()][0]
    lon_name = [v for v in ds.variables if 'lon' in v.lower()][0]

    # Read coordinates as NumPy arrays to limit memory overhead
    lats = ds[lat_name].values
    lons = ds[lon_name].values
    
    # Select grid cells inside the UK bounding box
    spatial_indices = np.where(
        (lons >= lon_min) & (lons <= lon_max) & 
        (lats >= lat_min) & (lats <= lat_max)
    )[0]
    
    # Collect feature arrays for the current file
    current_file_dict = {
        'lat': lats[spatial_indices],
        'lon': lons[spatial_indices]
    }
    
    # Broadcast the file timestamp to the selected grid cells
    if 'time' in ds.variables:
        times = ds['time'].values
        # If time is an array, take the first element as the file timestamp
        current_time = times[0] if times.ndim > 0 else times
    else:
        current_time = np.nan
    current_file_dict['time'] = np.repeat(current_time, len(spatial_indices))
    
    # Extract physical fields, accounting for 1-D, 2-D or 3-D layouts
    for var in var_list:
        if var in ds.variables and var not in [lat_name, lon_name]:
            var_data = ds[var].values
            if var_data.ndim == 1:
                current_file_dict[var] = var_data[spatial_indices]
            elif var_data.ndim == 2:
                current_file_dict[var] = var_data[-1, spatial_indices]
            elif var_data.ndim == 3:
                current_file_dict[var] = var_data[-1, -1, spatial_indices]
                
    ds.close()
    
    # Convert the current file to a DataFrame and append it
    df_step = pd.DataFrame(current_file_dict)
    processed_dataframes.append(df_step)
    
    print(f"Successfully extracted variables from [{idx+1}/{len(nc_files)}]: {filename}")


# Concatenate files along time and write the table

print("\n--- [Step 3] Concatenating datasets along time axis ---")

# Concatenate grid-feature matrices from all time slices
final_dataframe_merged = pd.concat(processed_dataframes, ignore_index=True)

# Apply a consistent column order
final_dataframe_merged = final_dataframe_merged.sort_values(by=['time', 'lat', 'lon']).reset_index(drop=True)

print("\n>>> Final Merged UK Dataset (With Temperature & Area) Summary:")
print(final_dataframe_merged.info())

# Write a tabular file for subsequent analysis
output_csv_path = "./MUSICA_UK_Features_Merged.csv"
final_dataframe_merged.to_csv(output_csv_path, index=False)

print(f"\nDone! All {len(nc_files)} datasets merged sequentially with all requested features.")
print(f"--> Saved to: {output_csv_path}")


--- [Step 2] Commencing Spatial Extraction for UK (Comprehensive Variables) ---
Successfully extracted variables from [1/15]: MUSICA_production3.2_BB.cam.h2.2018-07-01-03600.nc
Successfully extracted variables from [2/15]: MUSICA_production3.2_BB.cam.h2.2018-07-02-03600.nc
Successfully extracted variables from [3/15]: MUSICA_production3.2_BB.cam.h2.2018-07-03-03600.nc
Successfully extracted variables from [4/15]: MUSICA_production3.2_BB.cam.h2.2018-07-04-03600.nc
Successfully extracted variables from [5/15]: MUSICA_production3.2_BB.cam.h2.2018-07-05-03600.nc
Successfully extracted variables from [6/15]: MUSICA_production3.2_BB.cam.h2.2018-07-06-03600.nc
Successfully extracted variables from [7/15]: MUSICA_production3.2_BB.cam.h2.2018-07-07-03600.nc
Successfully extracted variables from [8/15]: MUSICA_production3.2_BB.cam.h2.2018-07-08-03600.nc
Successfully extracted variables from [9/15]: MUSICA_production3.2_BB.cam.h2.2018-07-09-03600.nc
Successfully extracted variables from [10/15]: